In [ ]:
from ngsxditto.utils.loglevel import loggingSlider
loggingSlider(default_level="WARNING") # global level
loggingSlider("ngsxditto", default_level="DEBUG") # ngsxditto

In [ ]:
from ngsxditto import *
from ngsolve import *
from xfem import *
import ngsolve.webgui as ngw
from netgen.occ import *

In [ ]:
domain = MoveTo(-1, -1).Rectangle(2, 2).Face()
domain.edges.Max(X).name = "right"
domain.edges.Min(X).name = "left"
domain.edges.Min(Y).name = "bottom"
domain.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.2))

In [ ]:
dt = 4e-2
order = 2
t = Parameter(0)
starting_levelset = (x**2 + (y + 0.75)**2)**0.5 - 1/2
transport = ExplicitDGTransport(mesh, dt=dt, order=order, compile=False)
levelset = LevelSetGeometry(transport)
levelset.Initialize(starting_levelset)
ngw.Draw(levelset.field)

In [ ]:
fluid_params = FluidParameters(viscosity=0.2, surface_tension_coeff=0.5)

mean_curvature = MeanCurvatureSolver(mesh, order=order, lset=levelset)
mean_curvature.Step()
fluid = TaylorHood(mesh, fluid_params, lset=levelset, nitsche_stab=500, f=CF((0, -0.5)), surface_tension=mean_curvature.H, dt=dt, order=order + 1, ghost_stab=100, add_convection=True, add_number_space=False, time_order=2)
fluid.SetOuterBoundaryCondition(NitscheVelocityBC(region="right|left", values=CF((0, 0))))
fluid.SetOuterBoundaryCondition(NitscheNormalVelocityBC(region="bottom", values=CF(0)))
fluid.Initialize()

sol = fluid.SolveStokes()
ngw.Draw(IfPos(levelset.lsetp1, CF((0, 0)), sol.components[0]), mesh)

In [ ]:
velocity_extension = LevelsetBasedExtension(levelset, order=order, gamma=1e-3, dirichlet="top")

velocity_extension.SetRhs(fluid.gfu)
levelset.transport.SetWind(velocity_extension.field)

def should_finalize():
    return time_loop.i_inner == 3

end_time = 2

time_loop = TimeLoop(time=t, dt=dt, end_time=end_time, display_progress_bar=True, should_finalize=None)
time_loop.SetFinalizeRule(should_finalize)

cf_neg = Norm(fluid.gfu)
cf_pos = -1
animation = UnfittedNGSWebguiScene(levelset, cf_neg=cf_neg, cf_pos=cf_pos,
                                  #order=fluid.order, time=t, end_time=end_time,
                                  name="animation", min=-0.075, max=0.4, autoscale=False)

time_loop.Register(velocity_extension, name="vel ext.")
time_loop.Register(levelset, name="levelset")
time_loop.Register(mean_curvature, name="mean curvature")
time_loop.Register(fluid, name="moving stokes")
time_loop.Register(animation, name="animation")

time_loop()

In [ ]:
ngw.Draw(levelset.field)